In [ ]:
import pandas as pd

# Load 2090 samples
data_path = './2090_ready_samples.csv'
with open(data_path, 'r') as f:
     data_ccc = pd.read_csv(f)

data_ccc

In [ ]:
from openai import OpenAI
from together import Together
from tqdm import tqdm

# Get the count of items per line (each line is a list of numbers) in the annotations column into a list
all_annotations = []
annotations_count = []
for index, row in data_ccc.iterrows():
    annotations = row['annotations'].split(',')
    all_annotations.append(annotations)
    count = len(annotations)
    annotations_count.append(count)

# Create API client based on the specified API type
def create_client(API_type):
    if API_type == 'openai':
        key_file_path = './openai_api_key.txt'
        with open(key_file_path, 'r') as file:
            api_key = file.read().strip()
        client = OpenAI(api_key=api_key)

    elif API_type == 'together':
        key_file_path = './together_api_key.txt'
        with open(key_file_path, 'r') as file:
            api_key = file.read().strip()
        client = Together(api_key=api_key)
    return client

gpt_model = "gpt-4o-2024-08-06"
llama_model = "meta-llama/Llama-3-70B-chat-hf"

# samples
targets = data_ccc['target_compound'].tolist()
texts = data_ccc['text'].tolist()

# prompt: ori-en-CandR
def create_English_prompt_CandR(target, text): 
    part_1 = "In this annotation task, we ask you to judge whether you find the bolded term **"
    part_2 = """** contentious in the context of the surrounding text, delimited by triple quotes. For the purposes of this task, we consider a term to be contentious when you think it is potentially offensive, derogatory, hurtful, or otherwise inappropriate by current standards. In making your judgment, you might ask yourself, for example, whether you would use the term in a similar sentence and/or whether you would be surprised if the term were used in this way by others. If you find a term in the given sentence slightly contentious, answer 'contentious according to current standards'. If not, answer 'not contentious'. And give a reason.
    
\"\"\""""
    part_3 = """\"\"\""""
    return part_1 + target + part_2 + text + part_3


model_output = []
for x in tqdm(range(len(data_ccc))):
    print(x)
    user_prompt = create_English_prompt_CandR(targets[x], texts[x])  
    sample_output = []
    for y in range(annotations_count[x]):
        print(y)
        completion = create_client("openai").chat.completions.create(
        model = gpt_model,
        # model = llama_model,  # llama_model or gpt_model
        messages=[
            {
            "role": "user",
            "content": user_prompt}
        ],
        temperature=1 # 1 for more creative output
        )
        output = completion.choices[0].message.content
        print(output)
        sample_output.append(output)
    model_output.append(sample_output)

In [ ]:
# Convert output to numbers (0 for not contentious, 1 for contentious)
def English_label_to_number(label):
    if 'Not contentious' in label or 'not contentious' in label:
        return 0
    elif 'Contentious' in label or 'contentious' in label:
        return 1
    else:
        return label
    

converted_output = [
    [English_label_to_number(content) for content in sample_output]
    for sample_output in model_output
]

# add a new column  "score" to the ready_samples, which is the majority vote of the annotations
def majority_vote(proportion: float) -> float:
    """Get the majority vote."""
    if proportion == 0.5:
        return 0.5
    elif proportion > 0.5:
        return 1
    else:
        return 0


# make a new dataset including the new results
updated_data = data_ccc.copy()
updated_data['output_scores'] = converted_output
updated_data['output'] = model_output
updated_data['output_mv_score'] = updated_data['output_scores'].apply(lambda x: majority_vote(sum(x) / len(x)))

updated_data

In [ ]:
# Save output. Choose the output path of gpt or llama.

sav_file = "./gpt-4o temperature1 ori-en-CandR.csv"
# sav_file = "./llama-3-70b temperature1 ori-en-CandR.csv"
with open(sav_file, 'w') as f:
    updated_data.to_csv(f)